# Notebook 02: Processed to Curated
## Amazon Fine Food Reviews: Analytical Table Creation
This notebook reads cleaned Parquet data from the `/processed` zone and builds 
three analytical tables in the `/curated` zone, ready for querying via 
Synapse Serverless SQL.

In [0]:
# Configuration and storage connection
STORAGE_ACCOUNT = "stcw2bdcbh"
STORAGE_KEY = dbutils.secrets.get(scope="kv-scope", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

PROCESSED_PATH = f"abfss://processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/amazon-reviews/"
CURATED_PATH = f"abfss://curated@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

print("Configuration set.")
print(f"Processed path: {PROCESSED_PATH}")
print(f"Curated path: {CURATED_PATH}")

Configuration set.
Processed path: abfss://processed@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/
Curated path: abfss://curated@stcw2bdcbh.dfs.core.windows.net/


### Step 1: Load Processed Data
Read the partitioned Parquet files from the processed zone. Spark will automatically 
read all year partitions unless filtered.

In [0]:
# Load processed data
df_processed = spark.read.parquet(PROCESSED_PATH)

processed_count = df_processed.count()
print(f"Loaded {processed_count} rows from processed zone")
print(f"Columns: {df_processed.columns}")
df_processed.printSchema()

Loaded 568454 rows from processed zone
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Summary', 'Text', 'ReviewDate', 'ReviewMonth', 'HelpfulnessRatio', 'SentimentBand', 'ReviewYear']
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)
 |-- ReviewDate: date (nullable = true)
 |-- ReviewMonth: integer (nullable = true)
 |-- HelpfulnessRatio: double (nullable = true)
 |-- SentimentBand: string (nullable = true)
 |-- ReviewYear: integer (nullable = true)



### Step 2: Curated Table 1: Yearly Review Trends
Aggregates review volume and average score by year. This shows engagement 
and sentiment change over time.

In [0]:
# Curated Table 1: Yearly review trends
from pyspark.sql.functions import count, avg, round

df_yearly_trends = df_processed \
    .groupBy("ReviewYear") \
    .agg(
        count("Id").alias("TotalReviews"),
        round(avg("Score"), 2).alias("AvgScore"),
        count("UserId").alias("UniqueReviewers")
    ) \
    .orderBy("ReviewYear")

df_yearly_trends.show()

df_yearly_trends.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_PATH}yearly_review_trends/")

print("Curated table 1 written: yearly_review_trends")

+----------+------------+--------+---------------+
|ReviewYear|TotalReviews|AvgScore|UniqueReviewers|
+----------+------------+--------+---------------+
|      1999|           6|     5.0|              6|
|      2000|          32|    4.53|             32|
|      2001|          13|    3.54|             13|
|      2002|          73|    4.71|             73|
|      2003|         132|    4.33|            132|
|      2004|         561|    4.39|            561|
|      2005|        1335|    4.45|           1335|
|      2006|        6671|    4.31|           6671|
|      2007|       22300|    4.39|          22300|
|      2008|       34163|    4.35|          34163|
|      2009|       55326|     4.3|          55326|
|      2010|       85884|     4.2|          85884|
|      2011|      163299|    4.14|         163299|
|      2012|      198659|    4.12|         198659|
+----------+------------+--------+---------------+

Curated table 1 written: yearly_review_trends


#### Observation
Review volume increased from 6 reviews in 1999 to 198,659 in 2012 and the average 
score remained consistently between 4.1 and 4.5 throughout. The fact that scores stayed this stable as the platform scaled suggests the dataset skews heavily positive, which is typical of self-selected reviewers 
who are more motivated to leave feedback after a good experience rather than a bad one.

### Step 3: Curated Table 2: Helpfulness Rate by Score Band
Calculates the average helpfulness ratio for each sentiment band. This helps us understand 
whether positive or negative reviews are rated more useful by other customers.

In [0]:
# Curated Table 2: Helpfulness rate by score band
from pyspark.sql.functions import col, count, avg, round

df_helpfulness = df_processed \
    .filter(col("HelpfulnessRatio").isNotNull()) \
    .groupBy("SentimentBand", "Score") \
    .agg(
        count("Id").alias("TotalReviews"),
        round(avg("HelpfulnessRatio"), 4).alias("AvgHelpfulnessRatio")
    ) \
    .orderBy("Score")

df_helpfulness.show()

df_helpfulness.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_PATH}helpfulness_by_score_band/")

print("Curated table 2 written: helpfulness_by_score_band")

+-------------+-----+------------+-------------------+
|SentimentBand|Score|TotalReviews|AvgHelpfulnessRatio|
+-------------+-----+------------+-------------------+
|     Negative|    1|       40002|             0.5392|
|     Negative|    2|       19165|             0.5664|
|      Neutral|    3|       24217|             0.6255|
|     Positive|    4|       38639|             0.7908|
|     Positive|    5|      176379|             0.8715|
+-------------+-----+------------+-------------------+

Curated table 2 written: helpfulness_by_score_band


#### Observation
Positive reviews (Score 4-5) achieve a much higher average helpfulness 
ratio (0.79,0.87) than negative reviews (Score 1-2, ratio: 0.54,0.56). 
This suggests a bias towards positivity in helpfulness voting as readers are more likely to validate 
reviews that confirm a purchase decision. Any downstream model using helpfulness 
as a feature weight should normalise by score band to avoid 
underweighting critical feedback.

### Step 4: Curated Table 3: Top Reviewer Cohort Comparison
Identifies the most active reviewers and compares their average score against 
the overall dataset average, to determine whether these reviewers are more 
or less critical than typical users.

In [0]:
# Curated Table 3: Top reviewer cohort comparison
from pyspark.sql.functions import col, count, avg, round as spark_round
import builtins

overall_avg = df_processed.agg(avg("Score")).collect()[0][0]
print(f"Overall average score: {builtins.round(overall_avg, 2)}")

df_top_reviewers = df_processed \
    .groupBy("UserId") \
    .agg(
        count("Id").alias("ReviewCount"),
        spark_round(avg("Score"), 2).alias("AvgScore")
    ) \
    .filter(col("ReviewCount") >= 10) \
    .withColumn("VsOverallAvg", spark_round(col("AvgScore") - overall_avg, 2)) \
    .orderBy(col("ReviewCount").desc())

df_top_reviewers.show(10, truncate=True)

df_top_reviewers.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_PATH}top_reviewer_cohort/")

print(f"Curated table 3 written: top_reviewer_cohort")
print(f"Reviewers with 10+ reviews: {df_top_reviewers.count()}")

Overall average score: 4.18
+--------------------+-----------+--------+------------+
|              UserId|ReviewCount|AvgScore|VsOverallAvg|
+--------------------+-----------+--------+------------+
|d29d7848d996429bb...|        448|    4.54|        0.36|
|6d712877b5bdd457e...|        421|    4.49|        0.31|
|45df0013347f6005d...|        389|    4.65|        0.47|
|700861d24b6f7f897...|        365|    4.84|        0.66|
|4c6eea15ff044da5a...|        256|    4.45|        0.27|
|47c8179806e3c2192...|        204|    4.83|        0.65|
|a93ebd2a5fb272e0b...|        201|    3.75|       -0.43|
|44e10fb3f2ee7049e...|        199|     1.0|       -3.18|
|9dfbe2ed2aa1595a6...|        178|     4.6|        0.42|
|22a9c8072dc9cb60c...|        176|    3.95|       -0.23|
+--------------------+-----------+--------+------------+
only showing top 10 rows

Curated table 3 written: top_reviewer_cohort
Reviewers with 10+ reviews: 7590


#### Observation
The overall average score is 4.18. Within the top 10 most prolific reviewers, 
most score above this average. However there were two outliers: one reviewer 
with 199 reviews averaging 1.0 (impressively low average), and another with 201 
reviews averaging 3.75. These outliers could represent systematic negative 
reviewers, competitor activity, or bot behaviour that would warrant further 
investigation in a production pipeline. Furthermore, the same could be said 
about the positive outliers — without additional context it is difficult to 
draw fair conclusions about any individual reviewer's behaviour from score 
alone.

### Step 5: Final Validation & Summary
Row counts are logged for each curated table to confirm successful writes. 
This provides an auditable record of what was produced and where it landed.

In [0]:
# Final validation and summary
print("=== Notebook 02 Summary ===")
print(f"Input rows (processed): {processed_count}")
print(f"\nCurated tables written:")
print(f"  1. yearly_review_trends: {df_yearly_trends.count()} rows")
print(f"  2. helpfulness_by_score_band: {df_helpfulness.count()} rows")
print(f"  3. top_reviewer_cohort: {df_top_reviewers.count()} rows")
print("\nAll curated tables written successfully.")
print(f"Curated path: {CURATED_PATH}")

=== Notebook 02 Summary ===
Input rows (processed): 568454

Curated tables written:
  1. yearly_review_trends: 14 rows
  2. helpfulness_by_score_band: 5 rows
  3. top_reviewer_cohort: 7590 rows

All curated tables written successfully.
Curated path: abfss://curated@stcw2bdcbh.dfs.core.windows.net/
